In [45]:
import pandas as pd
import numpy as np

from scipy.stats import spearmanr

from xgboost import XGBRanker

import optuna
import json

In [46]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from scipy.stats import spearmanr
from xgboost import XGBRanker


In [47]:
monthly = pd.read_csv(
    r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\model_data.csv",
    parse_dates=["Date"]
)

with open(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processed\feature_cols.json") as f:
    feature_cols = json.load(f)

monthly = monthly.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

print("Shape:", monthly.shape, "  Features:", len(feature_cols))
print("Date range:", monthly["Date"].min().date(), "→", monthly["Date"].max().date())

Shape: (114554, 90)   Features: 79
Date range: 2017-01-31 → 2021-11-30


In [48]:
monthly["rank_target"] = (
    monthly.groupby("Date")["target_1m"]
    .transform(
        lambda x: pd.qcut(x.rank(method="first"), q=32, labels=False)
    )
    .astype(int)
)

In [49]:
# Minimum training window before the first fold
MIN_TRAIN_MONTHS = 36
# Number of months in each test fold
TEST_FOLD_MONTHS = 6
# Gap between training end and test start (months)
GAP_MONTHS = 1

all_months = sorted(monthly["Date"].dt.to_period("M").unique())

folds = []
fold_start_idx = MIN_TRAIN_MONTHS  # first test fold starts after MIN_TRAIN_MONTHS

while fold_start_idx + GAP_MONTHS < len(all_months):
    train_end_period = all_months[fold_start_idx - 1]          # last training month
    test_start_period = all_months[fold_start_idx + GAP_MONTHS]  # after gap
    test_end_idx = min(fold_start_idx + GAP_MONTHS + TEST_FOLD_MONTHS - 1, len(all_months) - 1)
    test_end_period = all_months[test_end_idx]

    folds.append({
        "train_end":   train_end_period.to_timestamp(how="end"),
        "test_start":  test_start_period.to_timestamp(how="start"),
        "test_end":    test_end_period.to_timestamp(how="end"),
    })

    fold_start_idx += TEST_FOLD_MONTHS  # advance by one test fold

print(f"{len(folds)} folds:")
for i, f in enumerate(folds):
    print(f"  Fold {i}: train ≤ {f['train_end'].date()}  |  "
          f"test {f['test_start'].date()} → {f['test_end'].date()}")

4 folds:
  Fold 0: train ≤ 2019-12-31  |  test 2020-02-01 → 2020-07-31
  Fold 1: train ≤ 2020-06-30  |  test 2020-08-01 → 2021-01-31
  Fold 2: train ≤ 2020-12-31  |  test 2021-02-01 → 2021-07-31
  Fold 3: train ≤ 2021-06-30  |  test 2021-08-01 → 2021-11-30


In [50]:
def evaluate_params(params):

    all_test_predictions = []

    for fold_idx, fold in enumerate(folds):

        train_df = monthly[monthly["Date"] <= fold["train_end"]].copy()

        test_df = monthly[
            (monthly["Date"] >= fold["test_start"]) &
            (monthly["Date"] <= fold["test_end"])
        ].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        val_cutoff = train_df["Date"].max() - pd.DateOffset(months=6)

        inner_train = train_df[train_df["Date"] <= val_cutoff]
        inner_valid = train_df[train_df["Date"] > val_cutoff]

        X_tr = inner_train[feature_cols]
        y_tr = inner_train["rank_target"]

        X_va = inner_valid[feature_cols]
        y_va = inner_valid["rank_target"]

        X_te = test_df[feature_cols]

        group_tr = inner_train.groupby("Date").size().tolist()
        group_va = inner_valid.groupby("Date").size().tolist()

        ranker = XGBRanker(**params)

        ranker.fit(
            X_tr,
            y_tr,
            group=group_tr,
            eval_set=[(X_tr, y_tr), (X_va, y_va)],
            eval_group=[group_tr, group_va],
            verbose=False,
        )

        test_df = test_df.copy()
        test_df["score"] = ranker.predict(X_te)

        all_test_predictions.append(
            test_df[
                ["Date", "SecuritiesCode", "target_1m", "score"]
            ]
        )

    oos = pd.concat(all_test_predictions, ignore_index=True)

    monthly_ic = (
        oos.groupby("Date")
           .apply(
               lambda x: spearmanr(
                   x["score"],
                   x["target_1m"]
               ).correlation,
               include_groups=False
           )
    )

    return monthly_ic.mean()

In [51]:
baseline_params = {

    "objective": "rank:pairwise",
    "eval_metric": "ndcg",

    "learning_rate": 0.05,
    "n_estimators": 1000,

    "max_depth": 6,
    "min_child_weight": 30,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "tree_method": "hist",
    "early_stopping_rounds": 50,
    "random_state": 42,

    "gamma": 0,
    "reg_alpha": 0,
    "reg_lambda": 1,
}

In [52]:
print(evaluate_params(baseline_params))

0.05724165622993693


In [53]:
def objective(trial):

    params = baseline_params.copy()

    params["learning_rate"] = trial.suggest_float(
        "learning_rate",
        0.03,
        0.08,
        log=True,
    )

    params["max_depth"] = trial.suggest_int(
        "max_depth",
        5,
        7,
    )

    params["min_child_weight"] = trial.suggest_int(
        "min_child_weight",
        20,
        40,
    )

    params["subsample"] = trial.suggest_float(
        "subsample",
        0.7,
        0.9,
    )

    params["colsample_bytree"] = trial.suggest_float(
        "colsample_bytree",
        0.7,
        0.9,
    )

    params["gamma"] = trial.suggest_float(
        "gamma",
        0,
        1,
    )

    params["reg_alpha"] = trial.suggest_float(
        "reg_alpha",
        0,
        1,
    )

    params["reg_lambda"] = trial.suggest_float(
        "reg_lambda",
        0.5,
        2,
    )

    return evaluate_params(params)

In [54]:
study = optuna.create_study(direction="maximize")
study.enqueue_trial({
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 30,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0,
    "reg_alpha": 0,
    "reg_lambda": 1,
})
study.optimize(
    objective,
    n_trials=150,
    show_progress_bar=True,
)

[I 2026-06-29 16:37:46,464] A new study created in memory with name: no-name-0ede3f6a-3529-491b-af67-f9212eeeba86


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-06-29 16:37:52,615] Trial 0 finished with value: 0.05724165622993693 and parameters: {'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 30, 'subsample': 0.8, 'colsample_bytree': 0.8, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 1}. Best is trial 0 with value: 0.05724165622993693.
[I 2026-06-29 16:38:00,375] Trial 1 finished with value: 0.033867597145350016 and parameters: {'learning_rate': 0.046116594321536405, 'max_depth': 5, 'min_child_weight': 23, 'subsample': 0.8954972714296613, 'colsample_bytree': 0.7835424493357034, 'gamma': 0.20177168302820425, 'reg_alpha': 0.889691264144012, 'reg_lambda': 1.2182551562344}. Best is trial 0 with value: 0.05724165622993693.
[I 2026-06-29 16:38:07,031] Trial 2 finished with value: 0.01575708535323866 and parameters: {'learning_rate': 0.05028662600502689, 'max_depth': 7, 'min_child_weight': 26, 'subsample': 0.8330463197478947, 'colsample_bytree': 0.7722179581796051, 'gamma': 0.04238382952133557, 'reg_alpha': 0.5315287527845415, 'reg_la

C:\Users\naksh\AppData\Local\Temp\ipykernel_42372\1368253364.py:58: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  lambda x: spearmanr(


[I 2026-06-29 16:38:45,455] Trial 9 finished with value: 0.02363459560722136 and parameters: {'learning_rate': 0.06464746390422081, 'max_depth': 7, 'min_child_weight': 32, 'subsample': 0.875612131891191, 'colsample_bytree': 0.7254757457446404, 'gamma': 0.9887167989630147, 'reg_alpha': 0.08269013105584766, 'reg_lambda': 0.9611533033846484}. Best is trial 0 with value: 0.05724165622993693.
[I 2026-06-29 16:38:51,130] Trial 10 finished with value: 0.014279373694206848 and parameters: {'learning_rate': 0.035673260467240335, 'max_depth': 6, 'min_child_weight': 40, 'subsample': 0.774904162065491, 'colsample_bytree': 0.7009391351450888, 'gamma': 0.38253043388444885, 'reg_alpha': 0.010746028076976832, 'reg_lambda': 1.4953524062863768}. Best is trial 0 with value: 0.05724165622993693.
[I 2026-06-29 16:39:00,575] Trial 11 finished with value: 0.03916960043713963 and parameters: {'learning_rate': 0.07792931536118812, 'max_depth': 6, 'min_child_weight': 35, 'subsample': 0.8480536668511193, 'colsam

C:\Users\naksh\AppData\Local\Temp\ipykernel_42372\1368253364.py:58: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  lambda x: spearmanr(


[I 2026-06-29 16:45:40,495] Trial 69 finished with value: 0.00921238404305869 and parameters: {'learning_rate': 0.055567194953487914, 'max_depth': 7, 'min_child_weight': 28, 'subsample': 0.8949997286026925, 'colsample_bytree': 0.7337763463848144, 'gamma': 0.9436132679831414, 'reg_alpha': 0.18646619315933521, 'reg_lambda': 0.8954128951828392}. Best is trial 48 with value: 0.06017792591615687.
[I 2026-06-29 16:45:45,419] Trial 70 finished with value: 0.02835322298652361 and parameters: {'learning_rate': 0.04372509449816534, 'max_depth': 6, 'min_child_weight': 32, 'subsample': 0.867980999454401, 'colsample_bytree': 0.8009268135458358, 'gamma': 0.3048101195507351, 'reg_alpha': 0.2792687130235929, 'reg_lambda': 1.4107445892143586}. Best is trial 48 with value: 0.06017792591615687.
[I 2026-06-29 16:45:51,771] Trial 71 finished with value: 0.05027691301410468 and parameters: {'learning_rate': 0.05048455653472397, 'max_depth': 6, 'min_child_weight': 29, 'subsample': 0.885658700990335, 'colsamp

In [55]:
study.best_params

{'learning_rate': 0.04775288242684678,
 'max_depth': 6,
 'min_child_weight': 28,
 'subsample': 0.8842897290090517,
 'colsample_bytree': 0.7428578893925195,
 'gamma': 0.0020352102944416195,
 'reg_alpha': 0.21181862347735053,
 'reg_lambda': 0.8170692724501276}

In [56]:
import json

with open(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\best_xgbranker_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)